In [ ]:
# Import modules
import paramiko
import os
import pandas as pd
import dotenv
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
from pprint import pprint
import json
from src import utils

In [ ]:
# Provide connection details in separate .env file for security
dotenv.load_dotenv()
hostname = os.getenv('HOSTNAME')  # Replace with your server's IP or hostname
user = os.getenv('USER')    # Replace with your SSH username    
password = os.getenv('PASSWORD')  # SSH password from .env file

PROJECT_DIRECTORY = '/mnt/data_raid/feierabend/AVL_FIRE/PEMWE/PEMStar_2'    # Project directory on the remote server
MODEL_NAME = 'PEMStar_BekaertPTL'                   # AVL FIRE model name
CASE_SET_NAME = 'PolCurve_Bek~HighMassFlow'        # Case set name within the model
CASE_NAME = None                                    # Set to None to search all cases, or specify a case name like 'Case_1'

# Or, if you want to search for files with specific names
# TARGET_FILENAMES = ['file1.txt', 'report.log']


In [ ]:
# --- SSH Client Initialization ---
ssh_client = paramiko.SSHClient()
# Automatically add the server's host key. For production, it's better to manage known_hosts explicitly.
ssh_client.set_missing_host_key_policy(paramiko.AutoAddPolicy())

try:
    # Connect to the SSH server
    ssh_client.connect(hostname=hostname, username=user, password=password)
    print(f"Connected to {hostname} using password.")

    # Open an SFTP client
    sftp_client = ssh_client.open_sftp()
    print(f"Opened SFTP session.")
except paramiko.AuthenticationException:
    print("Authentication failed. Check your username, password, or private key.")
except paramiko.SSHException as e:
    print(f"Could not establish SSH connection: {e}")  

# Retrieve Simulation 3D data
Get data from EnSight results export

In [ ]:
data_directory = 'results/3D_EnSight'
results_3d_data_paths = utils.retrieve_avl_fire_data_paths(
    sftp_client=sftp_client,
    project_directory=PROJECT_DIRECTORY,
    model_name=MODEL_NAME,
    case_set_name=CASE_SET_NAME,
    data_directory=data_directory,
)
# results_3d_data_paths[7]
results_3d_data_paths

In [ ]:
utils.sftp_get_dir(sftp_client, results_3d_data_paths[3], os.path.join('data', data_directory.split('.')[-1]))

In [ ]:
from src.ensight_to_xdmf import inspect_ensight_case
info = inspect_ensight_case(r"data\3D_EnSight\PEMStar_BekaertPTL_DOM_8_0.case")
info["time_values"][:]
info["cell_fields"][:10]


In [ ]:
from pathlib import Path
from src.ensight_to_xdmf import EnsightConversionConfig, convert_ensight_case

metadata = convert_ensight_case(
    EnsightConversionConfig(
        case_file=Path(r"data\3D_EnSight\PEMStar_BekaertPTL_DOM_8_0.case"),
        output_dir=Path(r"data\3D_EnSight_converted"),
        case_id="PEMStar_BekaertPTL_DOM_8_0",
    )
)

In [ ]:
from pathlib import Path
import pyvista as pv
xdmf_path = Path(r"data\3D_EnSight_converted\fields.xdmf")
reader = pv.get_reader(str(xdmf_path))
print(reader.time_values)
reader.set_active_time_point(1)
data = reader.read()
print(type(data))
print(data)

# If you want one mesh for plotting, combine the blocks first:
if isinstance(data, pv.MultiBlock):
    blocks = [block for block in data if block is not None and block.n_cells > 0]
    mesh = blocks[0] if len(blocks) == 1 else data.combine()
else:
    mesh = data
print(mesh)
print(mesh.cell_data.keys())


In [ ]:
# Then plot a slice:
field = "Flow_Temperature"
slice_xy = mesh.slice(normal="x")
plotter = pv.Plotter()
plotter.add_mesh(slice_xy, scalars=field, cmap="viridis")
plotter.show(jupyter_backend="static")

In [ ]:
# Alternative: clip the volume
clipped = mesh.clip(normal="z", invert=False)
plotter = pv.Plotter()
plotter.add_mesh(clipped, scalars="Flow_Temperature", cmap="plasma")
plotter.show()


In [ ]:
# For a vector field
# For example Flow_Velocity or Electromagnetics_ElectronicCurrentDensity.
# First inspect magnitude on a slice:
field = "Flow_Velocity"
mesh[f"{field}_mag"] = (mesh[field] ** 2).sum(axis=1) ** 0.5
slice_xy = mesh.slice(normal="y")
plotter = pv.Plotter()
plotter.add_mesh(slice_xy, scalars=f"{field}_mag", cmap="turbo")
plotter.show()


In [ ]:
# Then, if needed, add arrows on a reduced sample:
# sampled = slice_xy.cell_centers().glyph(orient="Flow_Velocity", factor=0.001)
plotter = pv.Plotter()
plotter.add_mesh(slice_xy, scalars="Flow_Velocity_mag", opacity=0.6)
plotter.add_mesh(sampled, color="black")
plotter.show()